# 03 · Qualidade – Validações da Silver (Fabric)

Valida as tabelas Silver antes da Gold e gera um relatório com **PASS / WARN / FAIL**.

- **FAIL**: erro que invalida o dado (chave nula, duplicidade, tipo errado, valor fora do domínio). A Gold não deve rodar.
- **WARN**: situação esperada ou conhecida, que precisa de decisão documentada (ex.: município sem PIB, anos sem estimativa).
- **PASS**: regra atendida.

Saídas:
- Tabela `qualidade_resultados` (histórico de cada execução).
- Relatório em `Files/qualidade/relatorio_qualidade.md` (baixar e salvar em `docs/`).

Decisão de escopo (premissa 10): a Gold usa **só os anos presentes nas duas fontes**.

In [ ]:
import os
from datetime import datetime, timezone
from pyspark.sql import functions as F

pop = spark.table("silver_populacao")
pib = spark.table("silver_pib")
mun = spark.table("silver_municipio")

EXECUCAO = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
ANO_MIN, ANO_MAX = 2001, datetime.now().year
FILES = "/lakehouse/default/Files"

resultados = []   # um item por regra
detalhes = {}     # exemplos de ocorrências, para o relatório

def checar(id_, categoria, tabela, regra, ocorrencias, nivel_se_falhar="FAIL", exemplos=None):
    status = "PASS" if ocorrencias == 0 else nivel_se_falhar
    resultados.append({"id": id_, "categoria": categoria, "tabela": tabela, "regra": regra,
                       "status": status, "ocorrencias": int(ocorrencias)})
    if exemplos is not None and ocorrencias > 0:
        detalhes[id_] = exemplos
    print(f"[{status:4}] {id_} {regra}: {ocorrencias}")

def amostra(df, n=10):
    """Exemplos em texto, para o relatório."""
    return [", ".join(f"{k}={v}" for k, v in r.asDict().items()) for r in df.limit(n).collect()]

In [ ]:
# 1. TIPOS: esquema esperado de cada tabela
esperado = {
    "silver_populacao": {"cod_municipio": "string", "ano": "int", "populacao": "bigint"},
    "silver_pib": {"cod_municipio": "string", "ano": "int",
                   "pib_mil_reais": "decimal(18,3)", "pib_reais": "decimal(21,3)"},
    "silver_municipio": {"cod_municipio": "string", "sigla_uf": "string", "nome_regiao": "string"},
}
for i, (tabela, cols) in enumerate(esperado.items(), 1):
    tipos = dict(spark.table(tabela).dtypes)
    erradas = [f"{c}: esperado {t}, veio {tipos.get(c, 'ausente')}" for c, t in cols.items() if tipos.get(c) != t]
    checar(f"T{i}", "Tipos", tabela, "Colunas com o tipo esperado", len(erradas), exemplos=erradas)

In [ ]:
# 2. NULOS
for tabela, df, col_valor in [("silver_populacao", pop, "populacao"), ("silver_pib", pib, "pib_mil_reais")]:
    sufixo = "pop" if tabela == "silver_populacao" else "pib"
    checar(f"N1_{sufixo}", "Nulos", tabela, "Chave (cod_municipio, ano) sem nulos",
           df.filter(F.col("cod_municipio").isNull() | F.col("ano").isNull()).count())
    checar(f"N2_{sufixo}", "Nulos", tabela, "Valor numérico presente quando a fonte informa número",
           df.filter((F.col("status_valor") == "ok") & F.col(col_valor).isNull()).count())

    # Valores "não disponível" na fonte: esperado para município antes da instalação
    sem_valor = (df.filter(F.col(col_valor).isNull())
                   .groupBy("cod_municipio")
                   .agg(F.min("ano").alias("de"), F.max("ano").alias("ate"), F.count("*").alias("anos"))
                   .join(mun.select("cod_municipio", "nome_municipio", "sigla_uf"), "cod_municipio", "left")
                   .orderBy("cod_municipio"))
    checar(f"N3_{sufixo}", "Nulos", tabela, "Valores 'não disponível' na fonte (município × ano)",
           df.filter(F.col(col_valor).isNull()).count(), "WARN", amostra(sem_valor, 20))

checar("N4", "Nulos", "silver_municipio", "Município com nome, UF e região",
       mun.filter(F.col("nome_municipio").isNull() | (F.col("nome_municipio") == "")
                  | F.col("sigla_uf").isNull() | F.col("nome_regiao").isNull()).count())

In [ ]:
# 3. DUPLICIDADE
for tabela, df, sufixo in [("silver_populacao", pop, "pop"), ("silver_pib", pib, "pib")]:
    dup = df.groupBy("cod_municipio", "ano").count().filter("count > 1")
    checar(f"D1_{sufixo}", "Duplicidade", tabela, "Uma linha por (cod_municipio, ano)",
           dup.count(), exemplos=amostra(dup))
dup_mun = mun.groupBy("cod_municipio").count().filter("count > 1")
checar("D2", "Duplicidade", "silver_municipio", "Uma linha por cod_municipio",
       dup_mun.count(), exemplos=amostra(dup_mun))

In [ ]:
# 4. DOMÍNIOS
for tabela, df, sufixo in [("silver_populacao", pop, "pop"), ("silver_pib", pib, "pib")]:
    checar(f"V1_{sufixo}", "Domínio", tabela, "Código do município com 7 dígitos",
           df.filter(~F.col("cod_municipio").rlike(r"^[0-9]{7}$")).count())
    checar(f"V2_{sufixo}", "Domínio", tabela, f"Ano entre {ANO_MIN} e {ANO_MAX}",
           df.filter((F.col("ano") < ANO_MIN) | (F.col("ano") > ANO_MAX)).count())
    checar(f"V3_{sufixo}", "Domínio", tabela, "Nenhum valor com formato inválido na fonte",
           df.filter(F.col("status_valor") == "invalido").count())

checar("V4", "Domínio", "silver_populacao", "População maior que zero",
       pop.filter(F.col("populacao") <= 0).count(), exemplos=amostra(pop.filter(F.col("populacao") <= 0)))
checar("V5", "Domínio", "silver_pib", "PIB maior ou igual a zero",
       pib.filter(F.col("pib_mil_reais") < 0).count())
checar("V6", "Domínio", "silver_pib", "Unidade do PIB sempre 'Mil Reais'",
       pib.filter(F.col("unidade") != "Mil Reais").count())
checar("V7", "Domínio", "silver_municipio", "UF válida (27 UFs pelo código IBGE)",
       mun.filter(F.col("sigla_uf").isNull()).count())
divergente = mun.filter(F.col("sigla_uf") != F.col("sigla_uf_nome"))
checar("V8", "Domínio", "silver_municipio", "UF pelo código igual à UF escrita no nome",
       divergente.count(), "WARN",
       amostra(divergente.select("cod_municipio", "nome_municipio", "sigla_uf", "sigla_uf_nome")))

In [ ]:
# 5. INTEGRIDADE ENTRE AS TABELAS
m_pop = pop.select("cod_municipio").distinct()
m_pib = pib.select("cod_municipio").distinct()
nomes = mun.select("cod_municipio", "nome_municipio", "sigla_uf")

so_pop = m_pop.subtract(m_pib).join(nomes, "cod_municipio", "left")
so_pib = m_pib.subtract(m_pop).join(nomes, "cod_municipio", "left")
checar("I1", "Integridade", "pop × pib", "Município na população sem nenhum PIB",
       so_pop.count(), "WARN", amostra(so_pop))
checar("I2", "Integridade", "pib × pop", "Município no PIB sem nenhuma população",
       so_pib.count(), "WARN", amostra(so_pib))

sem_cadastro = m_pop.union(m_pib).distinct().subtract(mun.select("cod_municipio"))
checar("I3", "Integridade", "fatos × silver_municipio", "Todo município dos fatos existe no cadastro",
       sem_cadastro.count(), exemplos=amostra(sem_cadastro))

anos_pop = {r.ano for r in pop.select("ano").distinct().collect()}
anos_pib = {r.ano for r in pib.select("ano").distinct().collect()}
ANOS_COMUNS = sorted(anos_pop & anos_pib)
checar("I4", "Integridade", "pib × pop", "Anos de PIB sem estimativa de população",
       len(anos_pib - anos_pop), "WARN", [str(sorted(anos_pib - anos_pop))])
checar("I5", "Integridade", "pop × pib", "Anos de população sem PIB",
       len(anos_pop - anos_pib), "WARN", [str(sorted(anos_pop - anos_pib))])

# Dentro dos anos em comum: um valor presente e o outro ausente (afeta o PIB per capita)
par = (pop.filter(F.col("ano").isin(ANOS_COMUNS)).select("cod_municipio", "ano", "populacao")
          .join(pib.filter(F.col("ano").isin(ANOS_COMUNS)).select("cod_municipio", "ano", "pib_mil_reais"),
                ["cod_municipio", "ano"], "full"))
incompleto = par.filter(F.col("populacao").isNull() != F.col("pib_mil_reais").isNull())
checar("I6", "Integridade", "pop × pib", "Nos anos em comum, população e PIB presentes juntos",
       incompleto.count(), "WARN", amostra(incompleto.join(nomes, "cod_municipio", "left").orderBy("cod_municipio", "ano")))
print(f"Anos em comum ({len(ANOS_COMUNS)}): {ANOS_COMUNS}")

In [ ]:
# 6. PLAUSIBILIDADE: população total do Brasil por ano entre 170 e 230 milhões
total = pop.groupBy("ano").agg(F.sum("populacao").alias("total")).orderBy("ano")
fora = total.filter((F.col("total") < 170_000_000) | (F.col("total") > 230_000_000))
checar("P1", "Plausibilidade", "silver_populacao", "População do Brasil por ano entre 170 e 230 milhões",
       fora.count(), "WARN", amostra(fora))
total.show(30)

In [ ]:
# RESULTADO: tabela de histórico + relatório em Markdown
df_res = (spark.createDataFrame(resultados)
            .withColumn("executado_em", F.lit(EXECUCAO))
            .select("executado_em", "id", "categoria", "tabela", "regra", "status", "ocorrencias"))
df_res.write.format("delta").mode("append").saveAsTable("qualidade_resultados")

contagem = {s: sum(1 for r in resultados if r["status"] == s) for s in ["PASS", "WARN", "FAIL"]}
linhas = [
    "# Relatório de qualidade – Panorama Municipal", "",
    f"Execução: {EXECUCAO}  ",
    f"Resumo: **{contagem['PASS']} PASS · {contagem['WARN']} WARN · {contagem['FAIL']} FAIL**", "",
    "| ID | Categoria | Tabela | Regra | Status | Ocorrências |",
    "|---|---|---|---|---|---|",
]
linhas += [f"| {r['id']} | {r['categoria']} | {r['tabela']} | {r['regra']} | {r['status']} | {r['ocorrencias']} |"
           for r in resultados]
linhas += ["", f"Anos usados na Gold (presentes nas duas fontes): {ANOS_COMUNS}", "", "## Detalhes (até 20 exemplos por regra)", ""]
for id_, exemplos in detalhes.items():
    linhas.append(f"**{id_}**")
    linhas += [f"- {e}" for e in exemplos]
    linhas.append("")

os.makedirs(f"{FILES}/qualidade", exist_ok=True)
with open(f"{FILES}/qualidade/relatorio_qualidade.md", "w", encoding="utf-8") as f:
    f.write("\n".join(linhas))

print(f"Resumo: {contagem}")
df_res.show(50, truncate=False)

# Portão de qualidade: com FAIL, a Gold não deve ser gerada
assert contagem["FAIL"] == 0, f"{contagem['FAIL']} regra(s) com FAIL: veja o relatório antes de seguir para a Gold."